# Course 09: MLOps Getting Started

**Companion notebook for**: `09-mlops-getting-started.html`

## Learning Objectives
- Set up the Vertex AI SDK and understand MLOps fundamentals
- Create and log experiment runs with parameters and metrics
- Register models in the Vertex AI Model Registry, manage versions and aliases
- Use the Metadata Store to log artifacts and query lineage
- Implement continuous training trigger patterns
- Compare model versions and select the best performer

## Prerequisites
- GCP project with Vertex AI API enabled
- Service account or user credentials with Vertex AI permissions
- Python 3.10+

---
## Setup & Dependencies

In [ ]:
%pip install -q google-cloud-aiplatform>=1.38.0 pandas matplotlib scikit-learn

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

# --- GCP Configuration ---
# Set these to your GCP project and region
PROJECT_ID = os.environ.get("GCP_PROJECT_ID", "your-project-id")
REGION = os.environ.get("GCP_REGION", "us-central1")
BUCKET_NAME = f"{PROJECT_ID}-mlops-demo"
BUCKET_URI = f"gs://{BUCKET_NAME}"

print(f"Project: {PROJECT_ID}")
print(f"Region:  {REGION}")
print(f"Bucket:  {BUCKET_URI}")

In [ ]:
from google.cloud import aiplatform

# Initialize Vertex AI SDK
aiplatform.init(
    project=PROJECT_ID,
    location=REGION,
    staging_bucket=BUCKET_URI,
)

print(f"Vertex AI SDK initialized.")
print(f"SDK version: {aiplatform.__version__}")

---
## Section 1: Vertex AI Experiments

Vertex AI Experiments provides a structured way to track, compare, and reproduce ML experiments.
Each experiment groups related **runs**, and each run captures:
- **Parameters**: hyperparameters, configuration values
- **Metrics**: evaluation scores (AUC, accuracy, F1, etc.)
- **Time-series metrics**: per-epoch training curves
- **Artifacts**: datasets, models, schemas

### Why This Matters for the Exam
- Section 5 tests your ability to track and compare experiments
- Know the difference between Experiments (comparison) and Metadata Store (lineage)

### 1.1 Create an Experiment and Log Runs

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, precision_score, recall_score

# Generate synthetic fraud detection data
X, y = make_classification(
    n_samples=5000,
    n_features=20,
    n_informative=12,
    n_redundant=4,
    n_classes=2,
    weights=[0.95, 0.05],  # Imbalanced (like real fraud data)
    random_state=42,
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {len(X_train)} (fraud: {y_train.sum()})")
print(f"Test samples:     {len(X_test)} (fraud: {y_test.sum()})")
print(f"Fraud ratio:      {y_train.mean():.2%}")

In [ ]:
# Create an experiment
EXPERIMENT_NAME = "fraud-detection-comparison"

aiplatform.init(
    project=PROJECT_ID,
    location=REGION,
    experiment=EXPERIMENT_NAME,
)

print(f"Experiment '{EXPERIMENT_NAME}' created/loaded.")

In [ ]:
# Define model configurations to compare
model_configs = [
    {
        "run_name": "logistic-regression-baseline",
        "model_class": LogisticRegression,
        "params": {"C": 1.0, "max_iter": 1000, "solver": "lbfgs"},
    },
    {
        "run_name": "random-forest-v1",
        "model_class": RandomForestClassifier,
        "params": {"n_estimators": 100, "max_depth": 10, "min_samples_split": 5},
    },
    {
        "run_name": "random-forest-v2-deep",
        "model_class": RandomForestClassifier,
        "params": {"n_estimators": 200, "max_depth": 20, "min_samples_split": 2},
    },
    {
        "run_name": "gradient-boosting-v1",
        "model_class": GradientBoostingClassifier,
        "params": {"n_estimators": 150, "max_depth": 5, "learning_rate": 0.1},
    },
    {
        "run_name": "gradient-boosting-v2-tuned",
        "model_class": GradientBoostingClassifier,
        "params": {"n_estimators": 300, "max_depth": 4, "learning_rate": 0.05},
    },
]

# Train each model and log to Vertex AI Experiments
trained_models = {}

for config in model_configs:
    run_name = config["run_name"]
    print(f"\n--- Running: {run_name} ---")

    with aiplatform.start_run(run_name) as run:
        # Log hyperparameters
        run.log_params({
            "model_type": config["model_class"].__name__,
            **{k: str(v) for k, v in config["params"].items()},
        })

        # Train model
        model = config["model_class"](**config["params"], random_state=42)
        model.fit(X_train, y_train)

        # Evaluate
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]

        metrics = {
            "accuracy": accuracy_score(y_test, y_pred),
            "auc_roc": roc_auc_score(y_test, y_prob),
            "f1_score": f1_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred),
            "recall": recall_score(y_test, y_pred),
        }

        # Log metrics
        run.log_metrics(metrics)

        trained_models[run_name] = {"model": model, "metrics": metrics}
        print(f"  AUC: {metrics['auc_roc']:.4f} | F1: {metrics['f1_score']:.4f}")

print("\nAll experiment runs logged successfully.")

### 1.2 Compare Experiment Runs

In [ ]:
# Retrieve experiment results as a DataFrame
experiment_df = aiplatform.get_experiment_df(EXPERIMENT_NAME)

# Display key columns sorted by AUC
display_cols = [
    "run_name",
    "param.model_type",
    "metric.auc_roc",
    "metric.f1_score",
    "metric.precision",
    "metric.recall",
]

# Filter to columns that exist
available_cols = [c for c in display_cols if c in experiment_df.columns]
results = experiment_df[available_cols].sort_values("metric.auc_roc", ascending=False)
print(results.to_string(index=False))

# Identify the best run
best_idx = experiment_df["metric.auc_roc"].idxmax()
best_run = experiment_df.loc[best_idx]
print(f"\nBest run: {best_run['run_name']} with AUC={best_run['metric.auc_roc']:.4f}")

In [ ]:
# Visualize experiment comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

run_names = [c["run_name"] for c in model_configs]
short_names = [n.replace("-", "\n") for n in run_names]

# AUC comparison
auc_scores = [trained_models[n]["metrics"]["auc_roc"] for n in run_names]
colors = ["#22c55e" if s == max(auc_scores) else "#3b82f6" for s in auc_scores]
axes[0].barh(short_names, auc_scores, color=colors)
axes[0].set_xlabel("AUC-ROC")
axes[0].set_title("AUC-ROC by Model", fontsize=13, fontweight="bold")
axes[0].set_xlim(min(auc_scores) - 0.05, 1.0)
for i, v in enumerate(auc_scores):
    axes[0].text(v + 0.002, i, f"{v:.4f}", va="center", fontsize=9)

# Multi-metric radar-style grouped bar
metric_names = ["accuracy", "f1_score", "precision", "recall"]
x = np.arange(len(metric_names))
width = 0.15
for i, name in enumerate(run_names):
    values = [trained_models[name]["metrics"][m] for m in metric_names]
    axes[1].bar(x + i * width, values, width, label=name.split("-")[0] + "...")

axes[1].set_xticks(x + width * 2)
axes[1].set_xticklabels(metric_names)
axes[1].set_ylabel("Score")
axes[1].set_title("Metrics Comparison", fontsize=13, fontweight="bold")
axes[1].legend(fontsize=8, loc="lower right")
axes[1].set_ylim(0, 1.1)

plt.tight_layout()
plt.show()

print("Green bar = best AUC model (champion candidate).")

---
## Section 2: Model Registry

The Vertex AI Model Registry is a **centralized repository** for managing trained model artifacts.
Key capabilities:
- **Version management**: each upload creates a new version
- **Version aliases**: human-readable labels like "champion" and "challenger"
- **Lineage**: connect models to the experiments and data that produced them
- **Deployment state**: track which versions are deployed to which endpoints

### Exam Relevance
- Know when to use Model Registry vs Metadata Store
- Understand version aliases for champion/challenger patterns

### 2.1 Register a Model

In [ ]:
import joblib
import tempfile

# Save the best model locally
best_model_name = best_run["run_name"] if isinstance(best_run, pd.Series) else list(trained_models.keys())[-1]
best_model = trained_models[best_model_name]["model"]
best_metrics = trained_models[best_model_name]["metrics"]

# Save model artifact
model_dir = tempfile.mkdtemp()
model_path = os.path.join(model_dir, "model.joblib")
joblib.dump(best_model, model_path)
print(f"Model saved to: {model_path}")
print(f"Model size: {os.path.getsize(model_path) / 1024:.1f} KB")

# Upload to GCS (required for Model Registry)
# In production, use: gsutil cp or google.cloud.storage
gcs_model_uri = f"{BUCKET_URI}/models/fraud-detector/{best_model_name}/"
print(f"Model artifact URI: {gcs_model_uri}")

In [ ]:
# Register model in Vertex AI Model Registry
registered_model = aiplatform.Model.upload(
    display_name="fraud-detector",
    artifact_uri=gcs_model_uri,
    serving_container_image_uri="us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-3:latest",
    labels={
        "team": "fraud-detection",
        "environment": "staging",
        "framework": "sklearn",
    },
    version_aliases=["champion", "v1"],
    version_description=(
        f"{best_model_name} | "
        f"AUC={best_metrics['auc_roc']:.4f} | "
        f"F1={best_metrics['f1_score']:.4f}"
    ),
)

print(f"Model registered: {registered_model.display_name}")
print(f"Resource name:    {registered_model.resource_name}")
print(f"Version ID:       {registered_model.version_id}")
print(f"Version aliases:  {registered_model.version_aliases}")

### 2.2 List and Compare Model Versions

In [ ]:
# List all versions of a model
model_list = aiplatform.Model.list(
    filter='display_name="fraud-detector"',
    order_by="create_time desc",
)

print(f"{'Version':>8}  {'Aliases':<25}  {'Description'}")
print("-" * 80)
for m in model_list:
    aliases = ", ".join(m.version_aliases) if m.version_aliases else "(none)"
    desc = m.version_description or "(no description)"
    print(f"{m.version_id:>8}  {aliases:<25}  {desc[:50]}")

print(f"\nTotal versions: {len(model_list)}")

In [ ]:
# Upload a second version (challenger) to demonstrate versioning
# Use the second-best model
sorted_models = sorted(
    trained_models.items(),
    key=lambda x: x[1]["metrics"]["auc_roc"],
    reverse=True,
)

challenger_name, challenger_data = sorted_models[1]
challenger_model = challenger_data["model"]
challenger_metrics = challenger_data["metrics"]

# Save and upload challenger
challenger_dir = tempfile.mkdtemp()
challenger_path = os.path.join(challenger_dir, "model.joblib")
joblib.dump(challenger_model, challenger_path)
challenger_gcs_uri = f"{BUCKET_URI}/models/fraud-detector/{challenger_name}/"

# Register as new version of same model
challenger_registered = aiplatform.Model.upload(
    display_name="fraud-detector",
    artifact_uri=challenger_gcs_uri,
    serving_container_image_uri="us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-3:latest",
    parent_model=registered_model.resource_name,
    version_aliases=["challenger", "v2"],
    version_description=(
        f"{challenger_name} | "
        f"AUC={challenger_metrics['auc_roc']:.4f} | "
        f"F1={challenger_metrics['f1_score']:.4f}"
    ),
)

print(f"Challenger registered as version: {challenger_registered.version_id}")
print(f"Aliases: {challenger_registered.version_aliases}")
print(f"\nChampion AUC:    {best_metrics['auc_roc']:.4f}")
print(f"Challenger AUC:  {challenger_metrics['auc_roc']:.4f}")

---
## Section 3: Metadata Store

The Vertex AI Metadata Store tracks **artifacts**, **executions**, and their relationships (lineage).
It answers critical questions:
- Which dataset was used to train this model?
- What pipeline produced this artifact?
- Which models were trained on data from a specific source?

### Key Concepts
- **Artifact**: a dataset, model, schema, or metric object
- **Execution**: a pipeline step, training job, or preprocessing run
- **Event**: an input or output relationship between artifact and execution
- **Context**: groups related artifacts and executions (e.g., an experiment)

### 3.1 Log Artifacts and Query Lineage

In [ ]:
from google.cloud.aiplatform.metadata import metadata

# Log an experiment run with artifact metadata
EXPERIMENT_NAME_META = "fraud-detection-lineage-demo"

aiplatform.init(
    project=PROJECT_ID,
    location=REGION,
    experiment=EXPERIMENT_NAME_META,
)

with aiplatform.start_run("lineage-demo-run-001") as run:
    # Log parameters including data source (creates lineage)
    run.log_params({
        "dataset_uri": "bq://project.dataset.fraud_features_2025q4",
        "dataset_version": "2025-12-01",
        "feature_count": "20",
        "training_samples": str(len(X_train)),
        "model_type": "GradientBoostingClassifier",
        "n_estimators": "300",
        "max_depth": "4",
        "learning_rate": "0.05",
    })

    # Log metrics
    run.log_metrics({
        "auc_roc": 0.962,
        "f1_score": 0.891,
        "precision": 0.923,
        "recall": 0.861,
        "training_time_seconds": 45.2,
    })

    # Log classification-specific metrics
    run.log_classification_metrics(
        labels=["legitimate", "fraud"],
        matrix=[[4650, 50], [35, 265]],  # confusion matrix
        fpr=[0.0, 0.01, 0.05, 0.10, 1.0],
        tpr=[0.0, 0.75, 0.88, 0.93, 1.0],
        threshold=[1.0, 0.8, 0.5, 0.3, 0.0],
    )

print("Run with full metadata logged successfully.")
print("Lineage: dataset -> training execution -> model artifact")

In [ ]:
# Query experiment data including metadata
lineage_df = aiplatform.get_experiment_df(EXPERIMENT_NAME_META)

print("Experiment DataFrame columns:")
for col in sorted(lineage_df.columns):
    print(f"  {col}")

print(f"\n--- Run Details ---")
for col in lineage_df.columns:
    val = lineage_df.iloc[0][col]
    if pd.notna(val):
        print(f"  {col}: {val}")

### 3.2 Query Artifacts and Lineage

In [ ]:
# List all artifacts in the metadata store
artifacts = aiplatform.Artifact.list(
    order_by="create_time desc",
)

print(f"Total artifacts in Metadata Store: {len(artifacts)}")
print(f"\n{'Display Name':<35}  {'Schema Title':<30}  {'State'}")
print("-" * 85)
for artifact in artifacts[:10]:  # Show first 10
    print(
        f"{(artifact.display_name or 'unnamed'):<35}  "
        f"{(artifact.schema_title or 'N/A'):<30}  "
        f"{artifact.state or 'N/A'}"
    )

In [ ]:
# List all executions in the metadata store
executions = aiplatform.Execution.list(
    order_by="create_time desc",
)

print(f"Total executions in Metadata Store: {len(executions)}")
print(f"\n{'Display Name':<35}  {'Schema Title':<30}  {'State'}")
print("-" * 85)
for execution in executions[:10]:
    print(
        f"{(execution.display_name or 'unnamed'):<35}  "
        f"{(execution.schema_title or 'N/A'):<30}  "
        f"{execution.state or 'N/A'}"
    )

print("\nEach execution represents a step in the ML pipeline.")
print("Artifacts are inputs/outputs of executions, forming a lineage DAG.")

---
## Section 4: Continuous Training Trigger Patterns

Continuous training (CT) automatically retrains models when conditions are met.
This is the core of Level 1+ MLOps maturity.

### Common Trigger Patterns
| Trigger | Implementation | Use Case |
|---|---|---|
| **Scheduled** | Cloud Scheduler -> Pipeline | Data arrives on a known cadence |
| **Data drift** | Model Monitoring -> Cloud Function -> Pipeline | Feature distributions shift |
| **Performance** | Evaluation step -> conditional pipeline | Accuracy drops below threshold |
| **Data volume** | BigQuery trigger -> Pipeline | Enough new labeled data |

Below we demonstrate a conceptual performance-triggered retraining pattern.

In [ ]:
# Conceptual continuous training trigger pattern
# This demonstrates the logic, not the full GCP infrastructure

class ContinuousTrainingTrigger:
    """Monitors model performance and triggers retraining."""

    def __init__(self, metric_name: str, threshold: float, pipeline_id: str):
        self.metric_name = metric_name
        self.threshold = threshold
        self.pipeline_id = pipeline_id
        self.history = []

    def evaluate(self, current_value: float) -> dict:
        """Check if retraining should be triggered."""
        self.history.append({
            "timestamp": datetime.now().isoformat(),
            "value": current_value,
        })

        should_retrain = current_value < self.threshold

        result = {
            "metric": self.metric_name,
            "current_value": current_value,
            "threshold": self.threshold,
            "should_retrain": should_retrain,
            "reason": (
                f"{self.metric_name} ({current_value:.4f}) dropped below "
                f"threshold ({self.threshold:.4f})"
                if should_retrain
                else f"{self.metric_name} ({current_value:.4f}) is above threshold"
            ),
        }

        if should_retrain:
            result["action"] = f"Triggering pipeline: {self.pipeline_id}"
            # In production: aiplatform.PipelineJob(...).submit()

        return result


# Simulate performance degradation over time
trigger = ContinuousTrainingTrigger(
    metric_name="auc_roc",
    threshold=0.92,
    pipeline_id="fraud-training-pipeline-v2",
)

# Simulate weekly AUC scores (model degrades over time)
simulated_scores = [0.962, 0.958, 0.951, 0.943, 0.935, 0.928, 0.919, 0.908]
weeks = [f"Week {i+1}" for i in range(len(simulated_scores))]

print(f"{'Week':<10} {'AUC':>8} {'Status':<12} {'Action'}")
print("-" * 65)

for week, score in zip(weeks, simulated_scores):
    result = trigger.evaluate(score)
    status = "RETRAIN" if result["should_retrain"] else "OK"
    action = result.get("action", "No action needed")
    print(f"{week:<10} {score:>8.4f} {status:<12} {action}")

In [ ]:
# Visualize performance degradation and trigger point
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(weeks, simulated_scores, "o-", color="#3b82f6", linewidth=2, markersize=8)
ax.axhline(y=0.92, color="#ef4444", linestyle="--", linewidth=2, label="Retrain Threshold (0.92)")

# Color points above/below threshold
for i, (week, score) in enumerate(zip(weeks, simulated_scores)):
    color = "#22c55e" if score >= 0.92 else "#ef4444"
    ax.scatter(i, score, color=color, s=100, zorder=5)
    ax.annotate(f"{score:.3f}", (i, score), textcoords="offset points",
                xytext=(0, 12), ha="center", fontsize=9)

# Mark the trigger point
trigger_week = next(i for i, s in enumerate(simulated_scores) if s < 0.92)
ax.axvline(x=trigger_week, color="#f59e0b", linestyle=":", alpha=0.7)
ax.annotate("Retraining\nTriggered", (trigger_week, 0.915),
            fontsize=11, color="#f59e0b", fontweight="bold",
            ha="center", va="top")

ax.set_xlabel("Time", fontsize=12)
ax.set_ylabel("AUC-ROC", fontsize=12)
ax.set_title("Continuous Training: Performance Degradation Trigger", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.set_ylim(0.89, 0.97)
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"Model performance naturally degrades as data distribution shifts.")
print(f"Automatic retraining triggers when AUC drops below {trigger.threshold}.")

### 4.1 Scheduled Pipeline Trigger

In [ ]:
# Creating a scheduled pipeline (production pattern)
# This shows the actual SDK calls for scheduling a Vertex AI Pipeline

# Step 1: Define the pipeline job
pipeline_job = aiplatform.PipelineJob(
    display_name="weekly-fraud-retrain",
    template_path="gs://my-bucket/pipelines/fraud-pipeline.yaml",
    pipeline_root=f"{BUCKET_URI}/pipeline-runs/",
    parameter_values={
        "training_data_uri": f"bq://{PROJECT_ID}.fraud_data.features",
        "threshold_auc": 0.92,
        "model_display_name": "fraud-detector",
    },
)

# Step 2: Create a recurring schedule
# Uncomment to actually create the schedule:
# schedule = pipeline_job.create_schedule(
#     display_name="weekly-fraud-retrain-schedule",
#     cron="0 2 * * 0",  # Every Sunday at 2 AM UTC
#     max_concurrent_run_count=1,
#     max_run_count=52,  # Run for 1 year
# )
# print(f"Schedule created: {schedule.display_name}")
# print(f"Cron: {schedule.cron}")

print("Schedule pattern defined (uncomment to create in production).")
print("Cron: '0 2 * * 0' = Every Sunday at 2:00 AM UTC")
print("\nIn production, this triggers the full ML pipeline:")
print("  1. Pull latest data from BigQuery")
print("  2. Validate data schema and distributions")
print("  3. Train new model")
print("  4. Evaluate against champion (deployment gate)")
print("  5. Deploy if metrics pass threshold")

---
## Section 5: Model Versioning and Comparison

A robust MLOps workflow requires systematic model version management.
This section demonstrates the champion/challenger pattern using the Model Registry.

In [ ]:
def compare_model_versions(champion_metrics: dict, challenger_metrics: dict,
                            thresholds: dict) -> dict:
    """
    Compare champion vs challenger model and decide whether to promote.

    Implements deployment gate logic:
    1. Challenger must meet absolute minimums
    2. Challenger must beat champion (within tolerance)
    3. Challenger must meet latency requirements
    """
    results = {
        "gates": [],
        "promote": True,
        "champion_metrics": champion_metrics,
        "challenger_metrics": challenger_metrics,
    }

    # Gate 1: Absolute minimum AUC
    gate1 = {
        "name": "Minimum AUC",
        "passed": challenger_metrics["auc_roc"] >= thresholds["min_auc"],
        "detail": f"Challenger AUC {challenger_metrics['auc_roc']:.4f} vs min {thresholds['min_auc']}",
    }
    results["gates"].append(gate1)

    # Gate 2: Must beat champion (within regression tolerance)
    auc_diff = challenger_metrics["auc_roc"] - champion_metrics["auc_roc"]
    gate2 = {
        "name": "Beat Champion",
        "passed": auc_diff >= -thresholds["regression_tolerance"],
        "detail": f"AUC diff: {auc_diff:+.4f} (tolerance: {thresholds['regression_tolerance']})",
    }
    results["gates"].append(gate2)

    # Gate 3: F1 minimum
    gate3 = {
        "name": "Minimum F1",
        "passed": challenger_metrics["f1_score"] >= thresholds["min_f1"],
        "detail": f"Challenger F1 {challenger_metrics['f1_score']:.4f} vs min {thresholds['min_f1']}",
    }
    results["gates"].append(gate3)

    # Overall decision
    results["promote"] = all(g["passed"] for g in results["gates"])
    return results


# Run the comparison
comparison = compare_model_versions(
    champion_metrics=best_metrics,
    challenger_metrics=challenger_metrics,
    thresholds={
        "min_auc": 0.90,
        "regression_tolerance": 0.01,  # Allow up to 1% regression
        "min_f1": 0.70,
    },
)

print("=== Deployment Gate Results ===")
print(f"\nChampion:   AUC={best_metrics['auc_roc']:.4f}  F1={best_metrics['f1_score']:.4f}")
print(f"Challenger: AUC={challenger_metrics['auc_roc']:.4f}  F1={challenger_metrics['f1_score']:.4f}")
print()
for gate in comparison["gates"]:
    status = "PASS" if gate["passed"] else "FAIL"
    print(f"  [{status}] {gate['name']}: {gate['detail']}")
print(f"\nDecision: {'PROMOTE challenger to champion' if comparison['promote'] else 'KEEP current champion'}")

In [ ]:
# If promotion is approved, update the version alias
if comparison["promote"]:
    print("Promoting challenger to champion...")
    print("In production, this would:")
    print("  1. Move 'champion' alias from v1 to v2")
    print("  2. Update endpoint traffic split")
    print("  3. Log the promotion in Metadata Store")
    print()

    # SDK calls for promotion (conceptual):
    # registered_model.versioning.add_version_aliases(
    #     version="v2", aliases=["champion"]
    # )
    # registered_model.versioning.remove_version_aliases(
    #     version="v1", aliases=["champion"]
    # )
    # endpoint.update_traffic({"v2": 100})

    print("Champion promotion completed.")
else:
    print("Challenger did not pass all gates. Keeping current champion.")
    print("Challenger remains available for analysis in Model Registry.")

---
## Summary

In this notebook we covered the core MLOps capabilities in Vertex AI:

1. **Vertex AI Experiments** -- Created experiments, logged hyperparameters and metrics across multiple model configurations, and compared runs to identify the best performer.

2. **Model Registry** -- Registered models with version aliases (champion/challenger), managed multiple versions, and demonstrated the promotion workflow.

3. **Metadata Store** -- Logged artifacts with full metadata, queried lineage, and understood how experiments data feeds into the broader metadata graph.

4. **Continuous Training** -- Implemented performance degradation triggers, visualized model decay over time, and demonstrated scheduled pipeline patterns.

5. **Model Versioning and Comparison** -- Built deployment gate logic that compares champion vs challenger models against minimum thresholds and regression tolerances.

### Exam Takeaways
- MLOps is about deploying **pipelines**, not models
- Know the three maturity levels (0: manual, 1: pipeline automation, 2: CI/CD)
- Experiments = comparison; Metadata Store = lineage
- Model Registry manages versions with aliases for champion/challenger patterns
- Continuous training requires triggers AND validation gates

**Next notebook**: Course 10 covers Vertex AI Feature Store for managing features at scale.